              ── INSTALL DEPENDENCIES ────

In [0]:
%pip install yfinance

             ── IMPORTS AND SPARK SESSION ────

In [0]:
import yfinance as yf
import requests
import pandas as pd
from datetime import datetime, timedelta
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, lit, max as spark_max
from delta.tables import DeltaTable

spark = SparkSession.builder.getOrCreate()
print(f"Spark version: {spark.version}")

             ── PIPELINE CONFIGURATION ────

In [0]:
# ── PIPELINE CONFIG ────────────────────────────────────────
CATALOG        = "zambia_macro_lakehouse"
BRONZE_SCHEMA  = f"{CATALOG}.bronze"
START_DATE     = "2014-01-01"
TODAY          = datetime.today().strftime("%Y-%m-%d")
RUN_TIMESTAMP  = datetime.now()

# ── TABLE NAMES ────────────────────────────────────────────
TBL_FX         = f"{BRONZE_SCHEMA}.fx"
TBL_COPPER     = f"{BRONZE_SCHEMA}.copper"
TBL_OIL        = f"{BRONZE_SCHEMA}.brent_crude"
TBL_INFLATION  = f"{BRONZE_SCHEMA}.inflation_cpi"
TBL_GDP        = f"{BRONZE_SCHEMA}.gdp"
TBL_LOG        = f"{BRONZE_SCHEMA}.pipeline_log"

# ── TICKERS ────────────────────────────────────────────────
TICKERS = {
    TBL_FX     : "ZMWUSD=X",
    TBL_COPPER : "HG=F",
    TBL_OIL    : "BZ=F",
}

print(f"Catalog        : {CATALOG}")
print(f"Bronze schema  : {BRONZE_SCHEMA}")
print(f"Run date       : {TODAY}")

           ── CELL 4: HELPER FUNCTIONS ────

In [0]:
def get_latest_date(table_name: str, date_col: str = "Date") -> str:
    """
    Returns the latest date in a Delta table.
    If table is empty or does not exist, returns global START_DATE.
    """
    try:
        latest = spark.table(table_name) \
                      .agg(spark_max(date_col).alias("latest")) \
                      .collect()[0]["latest"]
        if latest is None:
            return START_DATE
        # Move one day forward so we do not re-pull the last date
        next_date = (pd.Timestamp(latest) + timedelta(days=1)).strftime("%Y-%m-%d")
        print(f"  Latest date in {table_name.split('.')[-1]}: {latest} → pulling from {next_date}")
        return next_date
    except Exception as e:
        print(f"  Table {table_name} not found or empty. Using START_DATE.")
        return START_DATE


def pull_yfinance(ticker: str, start: str, end: str) -> pd.DataFrame:
    """
    Pulls daily OHLCV data from Yahoo Finance.
    Returns a clean flat pandas DataFrame.
    """
    df = yf.download(ticker, start=start, end=end, interval="1d", progress=False)
    if df.empty:
        print(f"  No new data for {ticker}")
        return pd.DataFrame()
    df.columns = [col[0] if isinstance(col, tuple) else col for col in df.columns]
    df = df.reset_index()
    df["source"] = "yahoo_finance"
    df["ticker"] = ticker
    return df


def merge_yfinance(df: pd.DataFrame, table_name: str, merge_key: str = "Date") -> int:
    """
    Merges new yfinance data into an existing Delta table.
    Inserts new rows only. Updates existing rows if data changed.
    Returns number of rows in the new batch.
    """
    if df.empty:
        print(f"  Skipping merge for {table_name.split('.')[-1]} — no new data.")
        return 0

    spark_df = spark.createDataFrame(df) \
                    .withColumn("ingestion_time", current_timestamp())

    delta_table = DeltaTable.forName(spark, table_name)

    delta_table.alias("target").merge(
        spark_df.alias("source"),
        f"target.{merge_key} = source.{merge_key}"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

    print(f"  Merged {len(df):,} rows into {table_name.split('.')[-1]}")
    return len(df)


def log_run(table_name: str, rows_processed: int, status: str, message: str = ""):
    """
    Writes a run record to the pipeline log Delta table.
    """
    log_entry = pd.DataFrame([{
        "run_timestamp" : RUN_TIMESTAMP,
        "table_name"    : table_name,
        "rows_processed": rows_processed,
        "status"        : status,
        "message"       : message,
        "run_date"      : TODAY
    }])

    spark.createDataFrame(log_entry) \
         .write.mode("append") \
         .format("delta") \
         .saveAsTable(TBL_LOG)


print("Helper functions loaded.")

         ── INGEST MARKET DATA (FX, COPPER, OIL) ────

In [0]:
print("=" * 55)
print("INGESTING MARKET DATA")
print("=" * 55)

for table_name, ticker in TICKERS.items():
    print(f"\nProcessing: {ticker}")
    try:
        from_date = get_latest_date(table_name)
        df = pull_yfinance(ticker, start=from_date, end=TODAY)
        rows = merge_yfinance(df, table_name)
        log_run(table_name, rows, "SUCCESS")
    except Exception as e:
        print(f"  ERROR: {e}")
        log_run(table_name, 0, "FAILED", str(e))

print("\nMarket data ingestion complete.")

            ── INGEST INFLATION (WORLD BANK) ────

In [0]:
print("=" * 55)
print("INGESTING INFLATION (WORLD BANK)")
print("=" * 55)

try:
    url = "https://api.worldbank.org/v2/country/ZM/indicator/FP.CPI.TOTL.ZG?format=json&per_page=20"
    response = requests.get(url).json()
    records = response[1]

    inflation_raw = pd.DataFrame([{
        "year"          : int(r["date"]),
        "inflation_pct" : r["value"],
        "indicator"     : r["indicator"]["value"],
        "country"       : r["country"]["value"],
        "source"        : "world_bank"
    } for r in records if r["value"] is not None])

    inflation_spark = spark.createDataFrame(inflation_raw) \
                           .withColumn("ingestion_time", current_timestamp())

    delta_table = DeltaTable.forName(spark, TBL_INFLATION)

    delta_table.alias("target").merge(
        inflation_spark.alias("source"),
        "target.year = source.year"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

    print(f"  Merged {len(inflation_raw):,} rows into inflation_cpi")
    log_run(TBL_INFLATION, len(inflation_raw), "SUCCESS")

except Exception as e:
    print(f"  ERROR: {e}")
    log_run(TBL_INFLATION, 0, "FAILED", str(e))

              ── INGEST INFLATION (WORLD BANK) ────

In [0]:
print("=" * 55)
print("INGESTING GDP (WORLD BANK)")
print("=" * 55)

try:
    url = "https://api.worldbank.org/v2/country/ZM/indicator/NY.GDP.MKTP.CD?format=json&per_page=20"
    response = requests.get(url).json()
    records = response[1]

    gdp_raw = pd.DataFrame([{
        "year"      : int(r["date"]),
        "gdp_usd"   : r["value"],
        "indicator" : r["indicator"]["value"],
        "country"   : r["country"]["value"],
        "source"    : "world_bank"
    } for r in records if r["value"] is not None])

    gdp_spark = spark.createDataFrame(gdp_raw) \
                     .withColumn("ingestion_time", current_timestamp())

    delta_table = DeltaTable.forName(spark, TBL_GDP)

    delta_table.alias("target").merge(
        gdp_spark.alias("source"),
        "target.year = source.year"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

    print(f"  Merged {len(gdp_raw):,} rows into gdp")
    log_run(TBL_GDP, len(gdp_raw), "SUCCESS")

except Exception as e:
    print(f"  ERROR: {e}")
    log_run(TBL_GDP, 0, "FAILED", str(e))

             ── BRONZE VALIDATION SUMMARY ───

In [0]:
print("=" * 55)
print("BRONZE LAYER — FINAL VALIDATION")
print("=" * 55)

tables = {
    TBL_FX        : "FX Rate (ZMW/USD)",
    TBL_COPPER    : "Copper (HG=F)",
    TBL_OIL       : "Brent Crude (BZ=F)",
    TBL_INFLATION : "Inflation CPI",
    TBL_GDP       : "GDP (World Bank)",
}

for table, label in tables.items():
    count = spark.table(table).count()
    print(f"{label:30s} → {count:,} rows")

print("=" * 55)
print(f"Pipeline log  → {spark.table(TBL_LOG).count():,} entries")
print("Bronze complete.")